In [1]:
import sys
import random
import gc, argparse
import copy
import time
import glfw
import configparser
from dotenv import load_dotenv
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from PIL import Image
import numpy as np
from datetime import datetime
import json
import os
from src.env.env_clr import RILAB_OMY_ENV
from src.controllers import load_controller 


/opt/miniconda3/envs/mujoco2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load experiment configuration and environment variables
load_dotenv() # Load .env

config = configparser.ConfigParser()
config.read('experiment.cfg')

# Load environment configuration
config_file_path = './configs/train_key_clr.json'
with open(config_file_path) as f:
    env_conf = json.load(f)
language_instruction = env_conf['language_instruction']
omy_env = RILAB_OMY_ENV(cfg=env_conf,
                        seed=None, 
                        action_type='delta_eef_pose', # Keep Delta EEF for teleop!
                        obs_type='joint_pos',         # Use Joints for observation!
                        vis_mode = 'keyboard',
                        build_mjcf=False)


omy_env.reset(leader_pose = True)
# Load keyboard controller
controller = load_controller('keyboard',env_conf)
controller.reset(omy_env)


-----------------------------------------------------------------------------
name:[tabletop_env] dt:[0.002] HZ:[500]
 n_qpos:[40] n_qvel:[39] n_qacc:[39] n_ctrl:[9]
 integrator:[IMPLICITFAST]

n_body:[35]
 [0/35] [world] mass:[0.00]kg
 [1/35] [vention_rail_carriage] mass:[22.53]kg
 [2/35] [ewellix_lift_higher_link] mass:[19.17]kg
 [3/35] [ewellix_lift_middle_link] mass:[15.59]kg
 [4/35] [shoulder_link] mass:[7.37]kg
 [5/35] [upper_arm_link] mass:[13.05]kg
 [6/35] [forearm_link] mass:[3.99]kg
 [7/35] [wrist_1_link] mass:[2.10]kg
 [8/35] [wrist_2_link] mass:[1.98]kg
 [9/35] [wrist_3_link] mass:[1.56]kg
 [10/35] [finger_1_link] mass:[0.05]kg
 [11/35] [finger_2_link] mass:[0.05]kg
 [12/35] [door] mass:[0.30]kg
 [13/35] [right_latch_pull] mass:[0.10]kg
 [14/35] [left_latch_pull] mass:[0.10]kg
 [15/35] [latch_lock] mass:[0.10]kg
 [16/35] [lorge/hatch_face] mass:[19.28]kg
 [17/35] [lorge/external_rotary_wheel] mass:[0.41]kg
 [18/35] [lorge/external_rotary_handle] mass:[0.03]kg
 [19/35] [lor

You can teleop your robot with keyboard
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```

In [3]:
NUM_TRIALS_PER_TASK =20
RESUME = False  # Set to True to resume recording into an existing dataset
DATASET_ROOT = config.get('experiment', 'DATASET_ROOT', fallback="./dataset/clr_teleoperation_dataset") #@param {type:"string"}


In [4]:
from src.dataset.utils import make_teleoperation_dataset

if os.path.exists(DATASET_ROOT):
    if RESUME:
        print("RESUME existing dataset")
        dataset = LeRobotDataset('temp', root=DATASET_ROOT)
        print(f"Loaded dataset with {dataset.num_episodes} existing episodes")
    else:
        import shutil
        print("REMOVE")
        shutil.rmtree(DATASET_ROOT)
        print("CREATE")
        dataset = make_teleoperation_dataset(DATASET_ROOT, state_dim=7)
else:
    print("CREATE")
    dataset = make_teleoperation_dataset(DATASET_ROOT, state_dim=7)

REMOVE
CREATE


In [5]:
episode_id = dataset.num_episodes if RESUME else 0
start_episode_id = episode_id
print(f"Starting from episode {episode_id}")

Starting from episode 0


In [6]:
while omy_env.env.is_viewer_alive() and episode_id < start_episode_id + NUM_TRIALS_PER_TASK:
    omy_env.step_env()
    if omy_env.env.loop_every(HZ=20):
        key_list = omy_env.env.get_key_pressed_list()
        done = omy_env.check_success()
        if done or 90 in key_list:  # 'z' key to reset
            print("END EPISODE")
            if done:
                dataset.save_episode()
                episode_id += 1
            else: 
                dataset.clear_episode_buffer()
                #pass
            omy_env.reset(leader_pose = True)
            current_joints = omy_env.get_observation()[:7].astype(np.float32)
            action = controller.get_action()
            omy_env.step(action)
            next_joints = omy_env.get_observation()[:7].astype(np.float32)
        
        current_joints = omy_env.get_observation()[:7].astype(np.float32)
        action = controller.get_action()
        omy_env.step(action)
        next_joints = omy_env.get_observation()[:7].astype(np.float32)
        
        agent_image, wrist_image, left_scene_image, right_scene_image = omy_env.grab_image()

        images = {"agent": agent_image, 
                  "wrist": wrist_image, 
                  "left_scene": left_scene_image, 
                  "right_scene": right_scene_image}
        
        for image_label in images.keys():
            if images[image_label] is not None:
                print(image_label)
                image = Image.fromarray(images[image_label])
                # resize to 256x256
                image = image.resize((256, 256))
                image = np.array(image)
                images[image_label] = image
        
        obj_states, recp_q_poses = omy_env.get_object_pose(pad=10)
        obj_poses = np.array(obj_states['poses'])
        
        # Add frame to the dataset
        dataset.add_frame( {
                "observation.image": images["agent"],
                "observation.wrist_image": images["wrist"],
                "observation.left_scene_image": images["left_scene"],
                "observation.right_scene_image": images["right_scene"],
                "observation.state": current_joints,
                "action": next_joints, # Absolute Joint action
                "observation.eef_pose": next_joints, # Absolute Joint observation
                'env.obj_pose': np.array(obj_states['poses'],dtype=np.float32),
                "env.obj_names": ','.join(obj_states['names']),
                "env.obj_q_names": ','.join(recp_q_poses['names']),
                "env.obj_q_states": np.array(recp_q_poses['poses'],dtype=np.float32),
                "env.config_file_name": config_file_path,
                "task": language_instruction
            }, 
        )


        last_obj_poses = obj_poses
        # based on the episode_id number, get the guide line
        omy_env.render(language_instruction, guideline= f' [Num Episode: {episode_id}/{NUM_TRIALS_PER_TASK}]')
    omy_env.env.sync_sim_wall_time()
omy_env.env.close_viewer()
dataset.finalize()

agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_sce

Map: 100%|██████████| 562/562 [00:00<00:00, 989.62 examples/s] 


DONE INITIALIZATION
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene


Map: 100%|██████████| 562/562 [00:00<00:00, 996.09 examples/s] 


DONE INITIALIZATION
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene


...

DONE INITIALIZATION
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene


Map: 100%|██████████| 726/726 [00:00<00:00, 1014.77 examples/s]


DONE INITIALIZATION
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene
agent
wrist
left_scene
right_scene


Map: 100%|██████████| 662/662 [00:00<00:00, 1037.47 examples/s]


DONE INITIALIZATION
agent
wrist
left_scene
right_scene


In [7]:
DATASET_REPO = config.get('experiment', 'DATASET_REPO', fallback="gimarchetti/clr-experiment-dataset") #@param {type:"string"}

In [8]:
# Refresh dataset with latest changes
!hf upload {DATASET_REPO} {DATASET_ROOT}  --repo-type=dataset

Start hashing 27 files.
Finished hashing 27 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...hunk-000/file-004.parquet:   1%|              |  526kB / 83.8MB            

Processing Files (0 / 1)      :   0%|              |  526kB / 1.41GB, 1.32MB/s  
New Data Upload               :   0%|              |  526kB /  334MB, 1.32MB/s  


  ...hunk-000/file-000.parquet:   1%|              |  554kB / 70.7MB            



  ...hunk-000/file-005.parquet:   1%|              |  525kB / 85.6MB            




  ...hunk-000/file-015.parquet:   1%|              |  496kB / 61.7MB            





  ...e-000020/frame-000000.png:   1%|              |   213B / 26.6kB            






  ...e-000020/frame-000000.png:   1%|              |   320B / 39.8kB            







  ...e-000020/frame-000000.png:   1%|              |   265B / 33.1kB            








  ...e-000020/frame-0

## In case of accidental deletion of the dataset
You can download the existing dataset

In [ ]:
!hf download {DATASET_REPO} --repo-type dataset --local-dir {DATASET_ROOT}